# Run Arnav's behavioral battery on Qwen (and any VLM) — one notebook

Clones `halli75/Algoverse`, **removes the Gemma-4 model lock** (word-for-word otherwise), **stages EMOTIC**
into the layout the runners expect, and runs a chosen SUCCESS experiment with `E2E_MODEL` set to Qwen /
Pixtral / Llama-3.2-Vision / Gemma. Run top-to-bottom. Heavy step is §3 (EMOTIC ~1GB download + build).

SUCCESS set: **exp03** cross-modal+dictator · **exp05** ultimatum · **exp06** present-bias ·
**exp09** XSTest over-refusal · **exp10** risk mediation.

## 0 · Install

In [ ]:
!pip -q install "transformers>=4.49" accelerate bitsandbytes gdown pandas pillow scikit-learn scipy opencv-python-headless
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1 · Auth (HF token required for gated models; XAI only for exp03/04 caption-rewrite control)

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ("HF_TOKEN","XAI_API_KEY"):
        try:
            v=userdata.get(k)
            if v: os.environ[k]=v
        except Exception: pass
except Exception: pass
os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", os.environ.get("HF_TOKEN",""))
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| XAI set:", bool(os.environ.get("XAI_API_KEY")))
if not os.environ.get("HF_TOKEN"): print("!! add HF_TOKEN in Colab secrets (needed to download Qwen/Gemma)")

## 2 · Clone + remove the model lock (inline patch — science untouched)

In [ ]:
import subprocess, re, glob
REPO="/content/halli75_algoverse"
if not os.path.isdir(REPO):
    subprocess.check_call(["git","clone","--depth","1","https://github.com/halli75/Algoverse.git",REPO])
GEMMA="google/gemma-4-E4B-it"
def patch(path):
    src=open(path,encoding="utf-8").read(); lines=src.splitlines(keepends=True); n=0
    for i,ln in enumerate(lines):
        if ln.strip()=="if mid != PRIMARY:":
            ind=ln[:len(ln)-len(ln.lstrip())]
            lines[i]=f"{ind}if False:  # model-lock removed (multimodel): E2E_MODEL selects the model\n"; n+=1
    out="".join(lines)
    out,n2=re.subn(r"env\[(['\"])E2E_MODEL['\"]\]\s*=\s*['\"]"+re.escape(GEMMA)+r"['\"]",
                   lambda m: f"env[{m.group(1)}E2E_MODEL{m.group(1)}] = os.environ.get({m.group(1)}E2E_MODEL{m.group(1)}, {m.group(1)}{GEMMA}{m.group(1)})", out)
    if n+n2 and out!=src: open(path,"w",encoding="utf-8").write(out)
    return n+n2
tot=sum(patch(p) for p in glob.glob(f"{REPO}/scripts/*.py"))
print("patched:", tot, "edit(s). Model lock removed -> E2E_MODEL selects the model.")
assert not [1 for p in glob.glob(f"{REPO}/scripts/*.py") if "if mid != PRIMARY:" in open(p,encoding="utf-8").read()], "a guard survived"

## 3 · Stage EMOTIC into the layout the runners expect

`EMOTIC_ROOT/emotic/<Folder>/<Filename>.jpg` + `EMOTIC_ROOT/emotic_pre/train.csv` + the pinned split.
Reuses Arnav's recipe: images via gdown, Annotations.zip from the repo, `train.csv` built by Tandon-A/`mat2py.py`.

In [ ]:
import base64, hashlib
EMOTIC_ROOT="/content/emotic_data"; os.makedirs(EMOTIC_ROOT, exist_ok=True)
csv_path=f"{EMOTIC_ROOT}/emotic_pre/train.csv"
if not (os.path.exists(csv_path) and os.path.exists(f"{EMOTIC_ROOT}/emotic")):
    # images (~1GB) from the pinned Drive id
    zp="/content/emotic_images.zip"
    if not (os.path.exists(zp) and os.path.getsize(zp)>1_000_000_000):
        subprocess.check_call(["gdown","https://drive.google.com/uc?id=1icMKzWIlmFKhTkb4OrH8QAHHaaGOP9Zo","-O",zp,"--fuzzy"])
    if not os.path.exists("/content/emotic_images/emotic"):
        os.makedirs("/content/emotic_images", exist_ok=True)
        subprocess.check_call(["bash","-lc",f"unzip -qo {zp} -d /content/emotic_images"])
    subprocess.check_call(["ln","-sfn","/content/emotic_images/emotic",f"{EMOTIC_ROOT}/emotic"])
    # Annotations.zip from the cloned repo -> symlink into EMOTIC_ROOT
    az=f"{REPO}/scripts/Annotations.zip"
    if not os.path.exists(az):  # fall back to the base64 copy in the repo
        b64=f"{REPO}/scripts/Annotations.zip.b64.txt"
        open("/content/Annotations.zip","wb").write(base64.b64decode(open(b64).read().encode())); az="/content/Annotations.zip"
    subprocess.check_call(["bash","-lc","rm -rf /content/annotations && mkdir -p /content/annotations && unzip -qo "+az+" -d /content/annotations"])
    import pathlib
    ann=next(pathlib.Path("/content/annotations").rglob("Annotations"), None); assert ann, "Annotations missing after unzip"
    subprocess.check_call(["ln","-sfn",str(ann),f"{EMOTIC_ROOT}/Annotations"])
    # build emotic_pre/train.csv via Tandon-A/emotic mat2py.py
    if not os.path.exists(csv_path):
        repo2="/content/emotic_repo"
        if not os.path.exists(repo2): subprocess.check_call(["git","clone","-q","https://github.com/Tandon-A/emotic.git",repo2])
        m=f"{repo2}/mat2py.py"; t=open(m,encoding="utf-8",errors="ignore").read()
        old=("      cv2.imwrite(os.path.join(save_dir, 'context1.png'), context_arr[-1])\n"
             "      cv2.imwrite(os.path.join(save_dir, 'body1.png'), body_arr[-1])")
        new=("      if generate_npy:\n"
             "        cv2.imwrite(os.path.join(save_dir, 'context1.png'), context_arr[-1])\n"
             "        cv2.imwrite(os.path.join(save_dir, 'body1.png'), body_arr[-1])")
        if old in t: open(m,"w",encoding="utf-8").write(t.replace(old,new))
        subprocess.check_call(["python","mat2py.py","--data_dir",EMOTIC_ROOT,"--label","all"], cwd=repo2)
else:
    print("EMOTIC already staged - skip")
n_jpg=sum(1 for _ in __import__("pathlib").Path(f"{EMOTIC_ROOT}/emotic").rglob("*.jpg"))
print("n_jpg =", n_jpg); assert n_jpg>=20000, f"incomplete unpack n_jpg={n_jpg}"
# pinned split ships in the repo
SPLIT=f"{REPO}/artifacts/colab/emotic_split_1e8ea1c22144dd9d.json"
sp=json.load(open(SPLIT)); h=hashlib.sha256(json.dumps(sp,sort_keys=True).encode()).hexdigest()[:16]
print("split hash", h, "| n_train", len(sp["train_ids"])); assert h=="1e8ea1c22144dd9d", "split hash mismatch"
print("EMOTIC staged at", EMOTIC_ROOT)

## 4 · Config — pick the model + experiment

In [ ]:
EXP   = "exp03"                               # exp03 / exp05 / exp06 / exp09 / exp10
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"         # or mistral-community/Pixtral-12B, meta-llama/Llama-3.2-11B-Vision-Instruct, google/gemma-4-E4B-it
TIER  = "smoke"                               # "smoke" (fast) or "full"

E2E_ROOT="/content/algoverse_run"; os.makedirs(f"{E2E_ROOT}/battery/locks", exist_ok=True)
env=dict(os.environ)
env.update({
  "E2E_MODEL": MODEL, "E2E_TIER": TIER,
  "E2E_ROOT": E2E_ROOT, "E2E_EMOTIC": EMOTIC_ROOT, "E2E_SPLIT": SPLIT,
  "E2E_LOCK": f"{E2E_ROOT}/battery/A100.lock",
  "PYTHONPATH": f"{REPO}/scripts" + (os.pathsep+os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""),
  "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})
os.makedirs(f"{E2E_ROOT}/battery/{EXP}", exist_ok=True)
print("will run", EXP, "on", MODEL, "| tier", TIER)
if EXP in ("exp03","exp04") and not env.get("XAI_API_KEY"):
    print("note: no XAI_API_KEY -> exp03/04 caption-rewrite control degrades to weak literal rewrite (rest is fine)")

## 5 · Run (streams the experiment log)

In [ ]:
import sys
runner=f"{REPO}/scripts/battery_{EXP}_run.py"
assert os.path.exists(runner), runner
p=subprocess.Popen([sys.executable,"-u",runner], cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout: print(line, end="")
rc=p.wait(); print("\n=== exit", rc, "===")

## 6 · Results

In [ ]:
import glob as _g
hits=sorted(_g.glob(f"{E2E_ROOT}/battery/{EXP}/results*.json"), key=os.path.getmtime, reverse=True)
if hits:
    res=json.load(open(hits[0])); print("results:", os.path.basename(hits[0]))
    print(json.dumps(res, indent=2, default=str)[:2500])
    print("\nmodel:", res.get("model") or res.get("model_id"), "| complete:", res.get("complete"))
else:
    print("no results.json under", f"{E2E_ROOT}/battery/{EXP}", "- check the log above")
# from google.colab import files; files.download(hits[0])